# Urdu translation + audiobook workshop
This notebook creates a **draft Urdu translation** and sentence-level MP3 files from source text you have permission to translate. It does not publish anything to the website. Review the Urdu carefully before generating audio or sharing it.

**Security:** add `OPENAI_API_KEY` through Colab Secrets (key icon). Never paste the key into a code cell or commit it to Git.

In [ ]:
from google.colab import userdata
import os, subprocess, sys, pathlib

os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
assert os.environ['OPENAI_API_KEY'], 'Add OPENAI_API_KEY in Colab Secrets first.'

REPO_URL = 'https://github.com/mahashemi/BookReadingFramework.git'
PIPELINE_BRANCH = 'feature/urdu-translation-pipeline'  # Change to main after the PR is merged.
subprocess.run(['git','clone','--depth','1','--branch',PIPELINE_BRANCH,REPO_URL,'/content/BookReadingFramework'], check=True)
subprocess.run([sys.executable,'-m','pip','install','-r','/content/BookReadingFramework/tools/urdu_audiobook/requirements.txt'], check=True)
subprocess.run(['apt-get','-qq','update'], check=True)
subprocess.run(['apt-get','-qq','install','-y','ffmpeg'], check=True)
print('Setup complete.')

## 1. Choose model IDs and voice
These are configuration defaults, not a guarantee that a model is enabled for your API account. Check current model availability and pricing before large runs. Test a short excerpt first.

In [ ]:
# Set these to model IDs available to your OpenAI API project.
os.environ['OPENAI_TRANSLATION_MODEL'] = 'gpt-4o'
os.environ['OPENAI_TTS_MODEL'] = 'gpt-4o-mini-tts'
os.environ['OPENAI_TTS_VOICE'] = 'marin'

from google.colab import files
uploaded = files.upload()  # Upload one authorized .txt or .md chapter at a time.
assert uploaded, 'Upload a source .txt or .md file.'
source_name = next(iter(uploaded))
source_path = pathlib.Path('/content') / source_name
output_dir = pathlib.Path('/content/urdu_audiobook_output')
output_dir.mkdir(exist_ok=True)
print('Source:', source_path, '| Output:', output_dir)

## 2. Translate, preserving sentence alignment
The pipeline stores each source sentence next to its Urdu translation. This alignment is the foundation for highlighting exactly the sentence being read. Translation is a draft; inspect the JSON before proceeding.

In [ ]:
pipeline = '/content/BookReadingFramework/tools/urdu_audiobook/pipeline.py'
subprocess.run([sys.executable, pipeline, 'translate', '--input', str(source_path), '--output-dir', str(output_dir)], check=True)
translation_path = output_dir / 'translation.json'
print('Draft saved:', translation_path)
files.download(str(translation_path))

## 3. Human review gate — required
Open and review `translation.json`. Check religious terminology, names and honorifics, quotations, references, omissions, and natural Urdu. **Do not run the next cell until you are satisfied with the translation.** For high-stakes religious material, have a fluent Urdu reader review it.

In [ ]:
# Deliberate safety gate: change to True only after reviewing translation.json.
APPROVE_TRANSLATION_FOR_AUDIO = False
assert APPROVE_TRANSLATION_FOR_AUDIO is True, 'Review translation.json, then explicitly set this gate to True.'
subprocess.run([sys.executable, pipeline, 'audio', '--translation', str(translation_path), '--output-dir', str(output_dir)], check=True)
print('Sentence MP3s, audio_manifest.json, review.html, and (if FFmpeg succeeds) chapter.mp3 are ready.')

## 4. Download review bundle
The ZIP contains the aligned transcript, sentence MP3s, manifest, and a basic RTL review page. Audio is AI-generated and should be labelled as such when published. Do not upload generated files to the live website until editorial and pronunciation review is complete.

In [ ]:
subprocess.run([sys.executable, pipeline, 'package', '--output-dir', str(output_dir)], check=True)
files.download(str(output_dir / 'urdu_audiobook_bundle.zip'))